# RAG Assignment: Baseline Dense Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Regulation (EU) 2016/679 (General Data Protection Regulation - GDPR), *Protection of natural persons with regard to the processing of personal data and on the free movement of such data* (27 April 2016, OJ L 119).
It is a comprehensive 88-page normative legal regulation containing 173 recitals and 99 articles full of explicit obligations, rights, definitions, exceptions, and penalty thresholds, making it an ideal stress test for retrieval.

**Pipelines compared (same corpus, same Top-10 candidates, same LLM and prompt):**

| | Pipeline 1 – Baseline | Pipeline 2 – Re-ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS inner product → Top 10 | *same Top 10* |
| Stage 2 | keep dense ranks 1–3 | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores all 10 pairs → sort desc → Top 3 |
| Generation | Gemini (`gemini-3.1-flash-lite`), identical prompt → **Answer 1** | same → **Answer 2** |

Notebook outline: 1 setup · 2 PDF extraction · 3 chunking · 4 embeddings & index · 5 baseline retrieval · 6 re-ranking · 7 answer generation · 8 failure-case & rank-shift analysis · 9 latency · 10 conclusions & limitations.


## 1. Environment setup and imports

All models except the LLM run locally on CPU. The Gemini API key is read from the environment variable `GEMINI_API_KEY` (or a git-ignored `.env` file) — it is never written into this notebook.

In [1]:
import os, re, json, time, platform, statistics
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
import pymupdf                      # PyMuPDF, for PDF text extraction
import requests
import sentence_transformers
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Experiment configuration (everything that must stay identical across pipelines) ----
PDF_PATH         = Path("data/CELEX_32016R0679_EN_TXT.pdf")
FIRST_BODY_PAGE  = 1       # 1-based PDF page where body starts; GDPR starts from page 1
MAX_CHUNK_TOKENS = 200     # measured with the bi-encoder tokenizer (model limit is 256 word pieces -> no truncation)
OVERLAP_TOKENS   = 40      # trailing sentences (<= 40 tokens) repeated at the start of the next chunk in the same section
BI_ENCODER_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10                 # candidates from dense retrieval
TOP_N = 3                  # chunks given to the LLM
LLM_MODEL = "gemini-3.1-flash-lite"   # free tier allows only ~20 requests/day for gemini-3.8-flash; a run needs 32
GENERATION_CONFIG = {"temperature": 0.0, "maxOutputTokens": 1024, "thinkingConfig": {"thinkingLevel": "low"}}
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

torch.manual_seed(0); np.random.seed(0)
print("python", platform.python_version(), "| torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__,
      "| faiss", faiss.__version__, "| pymupdf", pymupdf.__version__)
print("CPU:", platform.processor() or platform.machine(), "| logical cores:", os.cpu_count(), "| torch threads:", torch.get_num_threads(),
      "| CUDA:", torch.cuda.is_available())

python 3.10.12 | torch 2.14.1+cpu | sentence-transformers 6.1.0 | faiss 1.15.1 | pymupdf 1.28.2
CPU: x86_64 | logical cores: 12 | torch threads: 8 | CUDA: False


In [2]:
# Load GEMINI_API_KEY from a local .env file if it is not already set (tiny parser, avoids an extra dependency).
if "GEMINI_API_KEY" not in os.environ and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
HAVE_LLM = bool(os.environ.get("GEMINI_API_KEY"))
print("GEMINI_API_KEY found:", HAVE_LLM, "(retrieval and latency cells still run without it; generation is skipped)")

GEMINI_API_KEY found: True (retrieval and latency cells still run without it; generation is skipped)


## 2. PDF loading and text extraction

* Every page starts with the same running header (title lines, page number, "This publication is available free of charge…"). Everything up to and including that line is dropped.
* Section headings come from the PDF's own bookmark outline (`get_toc()`), because Word's automatic heading numbers (e.g. "4.1") are not in the extracted text. Each heading line found on its page is tagged so that every chunk can be traced back to its section.

In [3]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"{PDF_PATH} missing - download GDPR from https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679")

doc = pymupdf.open(PDF_PATH)

# EUR-Lex running header pattern (page numbers, journal refs like "L 119/2")
RUNNING_HDR = re.compile(r"^(L\s*\d+/\d+|\d+\s*/\d+|EN\s+Official|Official Journal|^\d{1,3}\s*$)")

def page_lines(page):
    """Return non-empty text lines with EUR-Lex running headers stripped."""
    raw = [l.strip() for l in page.get_text().splitlines()]
    return [l for l in raw if l and not RUNNING_HDR.match(l)]

CHAPTER_RE = re.compile(r"^CHAPTER\s+[IVX]+$")
ARTICLE_RE  = re.compile(r"^Article\s+(\d+)$")

pages = {p: page_lines(doc[p - 1]) for p in range(FIRST_BODY_PAGE, len(doc) + 1)}

empty = [p for p, ls in pages.items() if len(" ".join(ls)) < 30]
if empty:
    print(f"Warning: pages with very little text: {empty}")

total_chars = sum(len(" ".join(ls)) for ls in pages.values())
print(f"PDF pages: {len(doc)} | indexed pages: {FIRST_BODY_PAGE}-{len(doc)} ({len(pages)} pages) | characters: {total_chars:,}")


PDF pages: 88 | indexed pages: 1-88 (88 pages) | characters: 351,712


## 3. Chunking and chunk identification

1. Lines are joined into running text and split into sentence-like *units* (split after `.`, `;`, `:` when the next token starts a new sentence/bullet).
2. A heading always starts a new chunk, so a chunk never mixes two sections (consecutive headings such as "5.1" + "5.1.1" are merged).
3. Units are packed greedily up to **200 bi-encoder tokens**; the last ≤40 tokens of sentences are repeated at the start of the next chunk of the same section (overlap), so a rule split across a boundary is still seen whole once.
4. Each chunk gets a stable ID `C000…`, plus its section title and PDF page range — these IDs are carried through every later step.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
tok = bi_encoder.tokenizer
n_tokens = lambda s: len(tok.tokenize(s))

SENT_SPLIT = re.compile(r"(?<=[.;:])\s+(?=[A-Z\u2022(\[])")

def extract_units_gdpr(pages):
    """Parse GDPR pages into section-tagged sentence units using Article/Chapter headings."""
    units = []
    section = "Recitals"
    buf, buf_page = "", None

    def flush(final=False):
        nonlocal buf, buf_page
        parts = SENT_SPLIT.split(buf) if buf else []
        keep = parts if final else parts[:-1]
        for p in keep:
            if p.strip():
                units.append({"page": buf_page, "section": section, "text": p.strip(), "heading": False})
        buf = "" if final or not parts else parts[-1]

    for p, lines in pages.items():
        idx = 0
        while idx < len(lines):
            line = lines[idx]; idx += 1
            art_m = ARTICLE_RE.match(line)
            if art_m and idx < len(lines) and not ARTICLE_RE.match(lines[idx]):
                title_line = lines[idx]; idx += 1
                flush(final=True)
                section = f"Article {art_m.group(1)} -- {title_line}"
                units.append({"page": p, "section": section, "text": section, "heading": True})
                continue
            if CHAPTER_RE.match(line) and idx < len(lines):
                chap_title = lines[idx]; idx += 1
                flush(final=True)
                section = f"{line} -- {chap_title}"
                units.append({"page": p, "section": section, "text": section, "heading": True})
                continue
            if not buf:
                buf_page = p
            buf = line if not buf else (buf + line if buf.endswith("-") else buf + " " + line)
            before = len(units)
            flush(final=False)
            if len(units) > before:
                buf_page = p
        flush(final=True)
    return units

def split_long(text, limit):
    out, cur = [], []
    for w in text.split():
        if cur and n_tokens(" ".join(cur + [w])) > limit:
            out.append(" ".join(cur)); cur = []
        cur.append(w)
    if cur: out.append(" ".join(cur))
    return out

def make_chunks(units, max_tokens=MAX_CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    pieces = []
    for u in units:
        texts = split_long(u["text"], max_tokens - overlap) if n_tokens(u["text"]) > max_tokens - overlap else [u["text"]]
        for t in texts:
            pieces.append({**u, "text": t, "ntok": n_tokens(t)})
    chunks, cur = [], []

    def emit():
        chunks.append({"chunk_id": f"C{len(chunks):03d}", "section": cur[0]["section"],
                       "page_start": cur[0]["page"], "page_end": cur[-1]["page"],
                       "text": " ".join(x["text"] for x in cur)})

    for u in pieces:
        size = sum(x["ntok"] for x in cur)
        only_headings = all(x["heading"] for x in cur)
        if cur and ((u["heading"] and not only_headings) or size + u["ntok"] > max_tokens):
            emit()
            carry = []
            if not u["heading"]:
                for x in reversed(cur):
                    if x["heading"] or sum(y["ntok"] for y in carry) + x["ntok"] > overlap: break
                    carry.insert(0, x)
            cur = carry
        cur.append(u)
    if cur: emit()
    return chunks

units = extract_units_gdpr(pages)
chunks = make_chunks(units)
chunk_by_id = {c["chunk_id"]: c for c in chunks}
for c in chunks:
    c["n_tokens"] = n_tokens(c["text"])

assert len(chunk_by_id) == len(chunks)
assert max(c["n_tokens"] for c in chunks) <= bi_encoder.max_seq_length, "a chunk would be truncated"
lens = [c["n_tokens"] for c in chunks]
med = int(np.median(lens))
print(f"{len(units)} units -> {len(chunks)} chunks | tokens per chunk: min {min(lens)}, median {med}, max {max(lens)} (bi-encoder max_seq_length = {bi_encoder.max_seq_length})")
pd.DataFrame(chunks)[["chunk_id", "section", "page_start", "page_end", "n_tokens", "text"]].iloc[[0, 10, 50, 100, -1]]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1744 units -> 444 chunks | tokens per chunk: min 10, median 171, max 200 (bi-encoder max_seq_length = 256)


,chunk_id,section,page_start,page_end,n_tokens,text
0,C000,Recitals,1,1,191,I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27 April ...
10,C010,Recitals,3,3,200,(13) In order to ensure a consistent level of protection for natural persons throughout the Union and to prevent div...
50,C050,Recitals,10,10,200,further use; the nature of the personal data; the consequences of the intended further processing for data subjects;...
100,C100,Recitals,18,18,187,The processing of personal data should not be considered to be on a large scale if the processing concerns personal ...
443,C443,Article 99 -- Entry into force and application,87,88,108,Article 99 -- Entry into force and application 1. This Regulation shall enter into force on the twentieth day follow...


In [5]:
# Save the exact corpus so every rank reported later can be looked up.
with open(RESULTS_DIR / "chunks.jsonl", "w") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("wrote", RESULTS_DIR / "chunks.jsonl")

wrote results/chunks.jsonl


## 4. Dense embeddings and vector index (Bi-Encoder)

Each chunk is embedded **independently** by the bi-encoder into a 384-d vector. Vectors are L2-normalised, so the inner product used by `faiss.IndexFlatIP` equals cosine similarity. A flat index does exact (brute-force) search — fine for 246 chunks and it removes approximate-search error from the comparison.

In [6]:
t0 = time.perf_counter()
chunk_emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index_build_s = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
assert index.ntotal == len(chunks)
assert np.allclose(np.linalg.norm(chunk_emb, axis=1), 1.0, atol=1e-4)    # normalised -> IP == cosine
print(f"embedded {len(chunks)} chunks, dim={chunk_emb.shape[1]}, in {index_build_s:.2f}s (one-off, offline cost)")

embedded 444 chunks, dim=384, in 10.20s (one-off, offline cost)


## 5. Pipeline 1 – Baseline retrieval (Top 10 → original Top 3)

In [7]:
def dense_retrieve(query, k=TOP_K):
    # Stage 1 (shared by both pipelines): embed query, exact inner-product search, return Top-k in dense order.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    cands = [{"chunk_id": chunks[i]["chunk_id"], "dense_rank": r + 1, "dense_score": float(s)}
             for r, (i, s) in enumerate(zip(ids[0], scores[0])) if i != -1]
    if len(cands) < k:
        print(f"WARNING: only {len(cands)} candidates returned for {query!r}")
    return cands

def baseline_select(cands, n=TOP_N):
    # Pipeline 1: keep the original dense order and take the first n.
    return cands[:n]

def full_dense_rank(query):
    # Rank of every chunk in the whole corpus (diagnostic only: tells us how far away a missed chunk was).
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids = index.search(q, index.ntotal)
    return {chunks[i]["chunk_id"]: r + 1 for r, i in enumerate(ids[0])}

demo = dense_retrieve("Are password hints allowed?")
pd.DataFrame([{**c, "section": chunk_by_id[c["chunk_id"]]["section"]} for c in demo])

,chunk_id,dense_rank,dense_score,section
0,C059,1,0.236500,Recitals
1,C219,2,0.198299,"Article 12 -- Transparent information, communication and modalities for the exercise of the rights of the data"
2,C025,3,0.193451,Recitals
3,C231,4,0.189548,Article 14 -- Information to be provided where personal data have not been obtained from the data subject
4,C066,5,0.188624,Recitals
5,C051,6,0.181771,Recitals
6,C432,7,0.181442,Article 90 -- Obligations of secrecy
7,C272,8,0.177524,Article 32 -- Security of processing
8,C040,9,0.174083,Recitals
9,C275,10,0.169715,Article 34 -- Communication of a personal data breach to the data subject


## 6. Pipeline 2 – Cross-Encoder re-ranking (same Top 10 → re-sorted Top 3)

The cross-encoder reads the query and the chunk **together** (`[CLS] query [SEP] chunk [SEP]`) and outputs one relevance logit; **higher = more relevant** for this MS-MARCO model, so we sort descending. Python's sort is stable, so equal scores keep their dense order.

In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
CE_MAX_LEN = cross_encoder.max_seq_length

def rerank(query, cands, n=TOP_N):
    # Pipeline 2: score ALL candidate (query, chunk) pairs, sort by score descending, return (full re-ranked list, top n).
    pairs = [(query, chunk_by_id[c["chunk_id"]]["text"]) for c in cands]
    scores = cross_encoder.predict(pairs, batch_size=len(pairs), show_progress_bar=False)
    scored = [{**c, "ce_score": float(s)} for c, s in zip(cands, scores)]
    ranked = sorted(scored, key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(ranked):
        c["rerank_rank"] = r + 1
    return ranked, ranked[:n]

# Truncation check: query + longest chunk must fit in the cross-encoder window.
longest = max(chunks, key=lambda c: len(cross_encoder.tokenizer(c["text"])["input_ids"]))
pair_len = len(cross_encoder.tokenizer("a long example question about authenticators and assurance levels?", longest["text"])["input_ids"])
assert pair_len <= CE_MAX_LEN, f"pair length {pair_len} > cross-encoder max {CE_MAX_LEN}"
print(f"cross-encoder max_length={CE_MAX_LEN}; longest query+chunk pair = {pair_len} tokens -> no truncation")

ranked_demo, _ = rerank("Are password hints allowed?", demo)
pd.DataFrame([{**c, "section": chunk_by_id[c["chunk_id"]]["section"]} for c in ranked_demo])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross-encoder max_length=512; longest query+chunk pair = 214 tokens -> no truncation


,chunk_id,dense_rank,dense_score,ce_score,rerank_rank,section
0,C066,5,0.188624,-10.604587,1,Recitals
1,C059,1,0.236500,-10.830892,2,Recitals
2,C040,9,0.174083,-10.839788,3,Recitals
3,C025,3,0.193451,-10.873059,4,Recitals
4,C051,6,0.181771,-10.886968,5,Recitals
5,C219,2,0.198299,-11.006105,6,"Article 12 -- Transparent information, communication and modalities for the exercise of the rights of the data"
6,C272,8,0.177524,-11.080841,7,Article 32 -- Security of processing
7,C432,7,0.181442,-11.088972,8,Article 90 -- Obligations of secrecy
8,C275,10,0.169715,-11.102154,9,Article 34 -- Communication of a personal data breach to the data subject
9,C231,4,0.189548,-11.114752,10,Article 14 -- Information to be provided where personal data have not been obtained from the data subject


### Correctness checks for the two pipelines

In [9]:
def check_pipelines(query):
    cands = dense_retrieve(query)
    base = baseline_select(cands)
    ranked, rr = rerank(query, cands)
    ids = [c["chunk_id"] for c in cands]
    assert len(cands) == TOP_K and len(set(ids)) == TOP_K, "need 10 distinct candidates"
    assert [c["dense_rank"] for c in cands] == list(range(1, TOP_K + 1))
    assert all(a["dense_score"] >= b["dense_score"] - 1e-6 for a, b in zip(cands, cands[1:])), "baseline not in dense order"
    assert [c["chunk_id"] for c in base] == ids[:TOP_N], "baseline must be original top-3"
    assert sorted(c["chunk_id"] for c in ranked) == sorted(ids), "re-ranker must score exactly the same 10 candidates"
    assert all(a["ce_score"] >= b["ce_score"] for a, b in zip(ranked, ranked[1:])), "re-ranking must be descending"
    assert len(base) == len(rr) == TOP_N
    # Chunk IDs still point to the right text/score after sorting: recompute one CE score independently.
    c0 = ranked[0]
    s = float(cross_encoder.predict([(query, chunk_by_id[c0["chunk_id"]]["text"])], show_progress_bar=False)[0])
    assert abs(s - c0["ce_score"]) < 1e-3, "chunk id / score mismatch after sorting"
    # Determinism: stage 1 gives the same candidates when called again (both pipelines share them).
    assert [c["chunk_id"] for c in dense_retrieve(query)] == ids
    return True

print(all(check_pipelines(q) for q in ["Are password hints allowed?", "How long is an out-of-band authentication secret valid?"]))

True


## 7. LLM answer generation (identical prompt for both pipelines)

The only thing that differs between Answer 1 and Answer 2 is **which three chunks** fill the `{context}` slot (in each pipeline's own rank order). Model, temperature (0), thinking level, token limit and template are identical.

In [10]:
PROMPT_TEMPLATE = '''You are answering questions about GDPR (Regulation (EU) 2016/679 - General Data Protection Regulation).
Use ONLY the context passages below; do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not answer this question."
Keep normative keywords (SHALL, SHALL NOT, SHOULD, SHOULD NOT, MAY) as written, and cite the passage IDs you rely on in square brackets, e.g. [C012].
Answer in at most 4 sentences.

Context:
{context}

Question: {question}
Answer:'''

def build_prompt(question, selected):
    blocks = []
    for c in selected:
        ch = chunk_by_id[c["chunk_id"]]
        pages_ = f"p. {ch['page_start']}" + (f"-{ch['page_end']}" if ch["page_end"] != ch["page_start"] else "")
        blocks.append(f"[{ch['chunk_id']}] (Section {ch['section']}; PDF {pages_})\n{ch['text']}")
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=question)

GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL}:generateContent"
CACHE_PATH = RESULTS_DIR / "llm_cache.json"     # answers keyed by hash(model, config, prompt): re-runs and quota limits don't force new calls
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def cache_key(prompt):
    import hashlib
    return hashlib.sha256(json.dumps([LLM_MODEL, GENERATION_CONFIG, prompt], sort_keys=True).encode()).hexdigest()

def generate(prompt, max_retries=6):
    # Call Gemini (with retries on overload / per-minute limits). Returns text, measured latency, model version, error.
    key = cache_key(prompt)
    if key in llm_cache:
        return {**llm_cache[key], "cached": True}
    if not HAVE_LLM:
        return {"text": None, "error": "GEMINI_API_KEY not set", "latency_s": None}
    body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}], "generationConfig": GENERATION_CONFIG}
    last_err = None
    for attempt in range(1, max_retries + 1):
        t0 = time.perf_counter()
        try:
            r = requests.post(GEMINI_URL, json=body, timeout=120,
                              headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
            latency = time.perf_counter() - t0
            if r.status_code == 200:
                j = r.json()
                parts = j["candidates"][0].get("content", {}).get("parts", [])
                text = "".join(p.get("text", "") for p in parts if not p.get("thought")).strip()
                out = {"text": text, "latency_s": latency, "model_version": j.get("modelVersion"),
                       "finish_reason": j["candidates"][0].get("finishReason"), "attempts": attempt, "error": None}
                llm_cache[key] = out
                CACHE_PATH.write_text(json.dumps(llm_cache, indent=1, ensure_ascii=False))
                return {**out, "cached": False}
            last_err = f"HTTP {r.status_code}: {r.text[:300]}"
            if r.status_code not in (429, 500, 502, 503, 504) or "PerDay" in r.text:
                break                                   # not retryable (bad request, auth, daily quota exhausted)
        except requests.RequestException as e:
            last_err = repr(e)
        time.sleep(min(60, 2 ** attempt))
    print("Generation failed:", last_err[:300])
    return {"text": None, "latency_s": None, "error": last_err, "attempts": attempt}

# Structural check: both prompts share the same template; only the context block differs.
_c = dense_retrieve("test question")
p1, p2 = build_prompt("Q?", baseline_select(_c)), build_prompt("Q?", rerank("Q?", _c)[1])
strip_ctx = lambda p: p.split("Context:")[0] + p.split("Question:")[1]
assert strip_ctx(p1) == strip_ctx(p2)
print(build_prompt("Can companies share user passwords with third parties?", baseline_select(demo))[:1200], "...")

You are answering questions about GDPR (Regulation (EU) 2016/679 - General Data Protection Regulation).
Use ONLY the context passages below; do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "The provided context does not answer this question."
Keep normative keywords (SHALL, SHALL NOT, SHOULD, SHOULD NOT, MAY) as written, and cite the passage IDs you rely on in square brackets, e.g. [C012].
Answer in at most 4 sentences.

Context:
[C059] (Section Recitals; PDF p. 11)
However, the controller should not refuse to take additional information provided by the data subject in order to support the exercise of his or her rights. Identification should include the digital identification of a data subject, for example through authentication mechanism such as the same credentials, used by the data subject to log-in to the on-line service offered by the data controller. (58) The principle of transparency requires that any information addressed to the public or

## 8. Failure-mode evaluation and rank-shift analysis

**How the queries were built.** I read the document and wrote questions in the way a user would ask them (often with everyday words such as *share passwords*, *subcontractor*, *delete account*, whereas GDPR uses legal terminology like *processing of personal data*, *another processor*, *erasure*). For each query I identified the relevant GDPR Article and provisions. Queries were refined after looking at retrieval behaviour, and the set deliberately includes varied categories (keyword traps, role limits, exceptions, negations, and buried chunks) as well as cases where re-ranking does and does not change the outcome.

`answer_check` is a regex for the key fact the correct answer must state; it provides a quick screen, supplemented by manual reading of the generated answers in §8.3.


In [11]:
QUERIES = [
    # --- Keyword trap / distractor ---
    {
        "qid": "Q01", "category": "keyword_trap",
        "query": "Can companies share user passwords with third parties?",
        "critical_chunk_hint": "Article 4 — Definitions",
        "note": "Query uses 'password' but GDPR uses 'personal data'; distractor chunks about data sharing use different terminology",
        "answer_check": r"(personal data|consent|prohibit|not allowed|lawfulness|security)"
    },
    {
        "qid": "Q02", "category": "role_limit",
        "query": "Can a data processor hire another subcontractor without permission?",
        "critical_chunk_hint": "Article 28 — Processor",
        "note": "Processor cannot engage another processor without prior specific or written authorization of the controller",
        "answer_check": r"(prior.*authorisation|written authorisation|not engage|consent|permission)"
    },
    # --- Exceptions & corner cases ---
    {
        "qid": "Q03", "category": "exception",
        "query": "Is processing sensitive data ever allowed without explicit consent?",
        "critical_chunk_hint": "Article 9 — Processing of special categories of personal data",
        "note": "Article 9(1) prohibits it; Article 9(2) lists exceptions (employment, vital interests, public health).",
        "answer_check": r"(exception|employment|vital interest|public health|legal claim|without consent)"
    },
    {
        "qid": "Q04", "category": "exception",
        "query": "Can a controller transfer personal data outside the EU?",
        "critical_chunk_hint": "Article 44 — General principle for transfers",
        "note": "Chapter V sets conditions: adequacy decision, appropriate safeguards, or binding corporate rules.",
        "answer_check": r"(adequacy|safeguards|binding corporate|third countr|condition)"
    },
    # --- Negation & constraints ---
    {
        "qid": "Q05", "category": "negation",
        "query": "What is explicitly prohibited when processing children data online?",
        "critical_chunk_hint": "Article 8 — Conditions applicable to child's consent",
        "note": "Consent for information society services offered directly to a child is only lawful if child is at least 16 (or lower member state age, min 13).",
        "answer_check": r"(16|13|parental|authoris|consent|below)"
    },
    {
        "qid": "Q06", "category": "negation",
        "query": "When is automated decision-making NOT allowed?",
        "critical_chunk_hint": "Article 22 — Automated individual decision-making",
        "note": "Data subject has right not to be subject to decision based solely on automated processing/profiling producing legal effects.",
        "answer_check": r"(legal effect|significantly affects|profiling|prohibit|exception)"
    },
    # --- Buried critical chunk ---
    {
        "qid": "Q07", "category": "buried_chunk",
        "query": "What is data protection by design and by default?",
        "critical_chunk_hint": "Article 25 — Data protection by design and by default",
        "note": "Article 25 mandates technical and organisational measures implementing data protection principles like minimisation by default.",
        "answer_check": r"(technical and organisational|measures|default|minimisation|design)"
    },
    {
        "qid": "Q08", "category": "role_limit",
        "query": "Who must appoint a Data Protection Officer?",
        "critical_chunk_hint": "Article 37 — Designation of the data protection officer",
        "note": "Mandatory for public authorities, core activities requiring regular/systematic monitoring, or large-scale special category data.",
        "answer_check": r"(public authority|core activities|regular and systematic monitoring|large scale)"
    },
    {
        "qid": "Q09", "category": "buried_chunk",
        "query": "What information must be provided when a person requests access to their data?",
        "critical_chunk_hint": "Article 15 — Right of access by the data subject",
        "note": "Article 15 specifies right to obtain confirmation, purposes, categories, recipients, and storage period.",
        "answer_check": r"(purposes|categories|recipients|storage period|copy)"
    },
    {
        "qid": "Q10", "category": "buried_chunk",
        "query": "What penalties exist for serious GDPR breaches?",
        "critical_chunk_hint": "Article 83 — General conditions for imposing administrative fines",
        "note": "Article 83(5) imposes fines up to 20,000,000 EUR or 4% of total worldwide annual turnover.",
        "answer_check": r"(20 000 000|20 million|4 %|4%|fine)"
    },
    # --- Additional queries ---
    {
        "qid": "Q11", "category": "keyword_trap",
        "query": "How long can companies keep customer data?",
        "critical_chunk_hint": "Article 5 — Principles relating to processing",
        "note": "Storage limitation principle: kept in a form which permits identification no longer than necessary for processing purposes.",
        "answer_check": r"(storage limitation|longer than is necessary|purposes)"
    },
    {
        "qid": "Q12", "category": "exception",
        "query": "Can personal data be processed for scientific research without consent?",
        "critical_chunk_hint": "Article 89 — Safeguards and derogations relating to processing for archiving",
        "note": "Article 89 provides derogations and safeguards like pseudonymisation for scientific or historical research purposes.",
        "answer_check": r"(safeguard|pseudonymisation|technical|derogation|research)"
    },
    {
        "qid": "Q13", "category": "negation",
        "query": "What rights can be restricted under national law?",
        "critical_chunk_hint": "Article 23 — Restrictions",
        "note": "Member state law may restrict rights for national security, defence, public security, or other public interests.",
        "answer_check": r"(national security|defence|public security|scope of obligations)"
    },
    {
        "qid": "Q14", "category": "role_limit",
        "query": "What breach notification obligations apply to processors?",
        "critical_chunk_hint": "Article 33 — Notification of a personal data breach",
        "note": "The processor must notify the controller without undue delay after becoming aware of a personal data breach.",
        "answer_check": r"(without undue delay|controller|notif)"
    },
    {
        "qid": "Q15", "category": "buried_chunk",
        "query": "Under what conditions can a supervisory authority impose a temporary ban on processing?",
        "critical_chunk_hint": "Article 58 — Powers of supervisory authorities",
        "note": "Article 58(2)(f) provides corrective power to impose a temporary or definitive limitation including a ban on processing.",
        "answer_check": r"(corrective power|ban|limitation|infringement)"
    },
    {
        "qid": "Q16", "category": "keyword_trap",
        "query": "Do users have the right to delete their account data?",
        "critical_chunk_hint": "Article 17 — Right to erasure",
        "note": "Query uses 'delete account'; GDPR Article 17 provides the 'Right to erasure' ('right to be forgotten').",
        "answer_check": r"(erasure|right to be forgotten|without undue delay|withdraw)"
    },
]
print(f"{len(QUERIES)} queries defined across categories: { {c: sum(1 for q in QUERIES if q['category']==c) for c in set(q['category'] for q in QUERIES)} }")


16 queries defined across categories: {'exception': 3, 'role_limit': 3, 'negation': 3, 'buried_chunk': 4, 'keyword_trap': 3}


### 8.1 Run both pipelines on every query

Stage 1 is executed **once** per query and its Top-10 list is passed to both pipelines, so they cannot see different candidates.

In [12]:
# Resolve critical_chunk_hint -> actual chunk IDs (runs after chunking)
import re as _re

def resolve_critical_chunks(queries, chunks):
    for qd in queries:
        hint = qd.get('critical_chunk_hint', '')
        art_m = _re.search(r'Article\s+(\d+)', hint)
        matched = []
        if art_m:
            art_num = art_m.group(1)
            # Match section title or first 150 chars of text
            matched = [c['chunk_id'] for c in chunks
                       if _re.search(rf'\bArticle\s+{art_num}\b', c['section'])
                       or _re.search(rf'\bArticle\s+{art_num}\b', c['text'][:150])]
        qd['critical'] = matched[:4] if matched else []
    return queries

QUERIES = resolve_critical_chunks(QUERIES, chunks)

# Show resolved critical chunk IDs for each query
for qd in QUERIES:
    secs = [chunk_by_id[c]['section'][:60] for c in qd['critical'][:1]]
    print(f"{qd['qid']} critical={qd['critical']} -> {secs}")


Q01 critical=['C184', 'C185', 'C186', 'C187'] -> ['Article 4 -- Definitions']
Q02 critical=['C178', 'C259', 'C260', 'C261'] -> ['Recitals']
Q03 critical=['C207', 'C208', 'C209', 'C210'] -> ['Article 9 -- Processing of special categories of personal da']
Q04 critical=['C311'] -> ['CHAPTER V -- Transfers of personal data to third countries o']
Q05 critical=['C205', 'C206'] -> ["Article 8 -- Conditions applicable to child's consent in rel"]
Q06 critical=['C224', 'C246', 'C247'] -> ['Article 13 -- Information to be provided where personal data']
Q07 critical=['C253', 'C254'] -> ['Article 25 -- Data protection by design and by default']
Q08 critical=['C286', 'C287', 'C288'] -> ['Article 37 -- Designation of the data protection officer']
Q09 critical=['C232', 'C233', 'C234'] -> ['Article 15 -- Right of access by the data subject']
Q10 critical=['C356', 'C415', 'C416', 'C417'] -> ['Article 58 -- Powers']
Q11 critical=['C194', 'C195', 'C196'] -> ['CHAPTER II -- Principles']
Q12 critical=['C429

In [13]:
runs, rank_rows = [], []
for qd in QUERIES:
    q = qd["query"]
    cands = dense_retrieve(q)                       # shared Top-10
    base_top = baseline_select(cands)               # Pipeline 1
    ranked, rr_top = rerank(q, cands)               # Pipeline 2
    full_rank = full_dense_rank(q)
    rr_rank = {c["chunk_id"]: c["rerank_rank"] for c in ranked}
    ce = {c["chunk_id"]: c["ce_score"] for c in ranked}
    for c in ranked:
        rank_rows.append({"qid": qd["qid"], "chunk_id": c["chunk_id"], "section": chunk_by_id[c["chunk_id"]]["section"],
                          "dense_rank": c["dense_rank"], "dense_score": round(c["dense_score"], 4),
                          "ce_score": round(c["ce_score"], 3), "rerank_rank": c["rerank_rank"],
                          "critical": c["chunk_id"] in qd["critical"]})
    crit = qd["critical"]
    best_dense = min((full_rank[c] for c in crit if c in full_rank), default=TOP_K + 1)
    in_top10 = [c for c in crit if c in rr_rank]
    runs.append({**qd, "cands": cands, "base_top": base_top, "rr_top": rr_top, "ranked": ranked,
                 "critical_dense_rank": {c: full_rank[c] for c in crit},
                 "critical_rerank_rank": {c: rr_rank.get(c) for c in crit},
                 "critical_ce": {c: round(ce[c], 2) for c in in_top10},
                 "base_hit": any(c["chunk_id"] in crit for c in base_top),
                 "rr_hit": any(c["chunk_id"] in crit for c in rr_top),
                 "best_dense_rank": best_dense})
    check_pipelines(q)

def outcome(r):
    if r["base_hit"] and r["rr_hit"]:   return "both hit"
    if r["rr_hit"]:                     return "RE-RANK RESCUE"
    if r["base_hit"]:                   return "RE-RANK LOSS"
    return "both miss (not in top 10)" if r["best_dense_rank"] > TOP_K else "both miss (in top 10)"

summary = pd.DataFrame([{
    "qid": r["qid"], "category": r["category"], "query": r["query"],
    "critical chunk: dense rank -> rerank rank": ", ".join(f"{c}: {r['critical_dense_rank'][c]} -> {r['critical_rerank_rank'][c] or '—'}"
                                                         for c in r["critical"]),
    "CE score": ", ".join(f"{c}: {s}" for c, s in r["critical_ce"].items()) or "—",
    "baseline top3": " ".join(c["chunk_id"] for c in r["base_top"]),
    "rerank top3": " ".join(c["chunk_id"] for c in r["rr_top"]),
    "critical in baseline top3": r["base_hit"], "critical in rerank top3": r["rr_hit"], "outcome": outcome(r)} for r in runs])
pd.DataFrame(rank_rows).to_csv(RESULTS_DIR / "rankings_top10.csv", index=False)
summary.to_csv(RESULTS_DIR / "rank_shift_summary.csv", index=False)
summary

,qid,category,query,critical chunk: dense rank -> rerank rank,CE score,baseline top3,rerank top3,critical in baseline top3,critical in rerank top3,outcome
0,Q01,keyword_trap,Can companies share user passwords with third parties?,"C184: 433 -> —, C185: 270 -> —, C186: 79 -> —, C187: 5 -> 1",C187: -7.24,C107 C059 C005,C187 C107 C005,False,True,RE-RANK RESCUE
1,Q02,role_limit,Can a data processor hire another subcontractor without permission?,"C178: 261 -> —, C259: 4 -> 1, C260: 20 -> —, C261: 199 -> —",C259: 0.59,C265 C127 C033,C259 C114 C262,False,True,RE-RANK RESCUE
2,Q03,exception,Is processing sensitive data ever allowed without explicit consent?,"C207: 79 -> —, C208: 19 -> —, C209: 21 -> —, C210: 198 -> —",—,C041 C028 C052,C040 C052 C028,False,False,both miss (not in top 10)
3,Q04,exception,Can a controller transfer personal data outside the EU?,C311: 5 -> 5,C311: 3.64,C120 C331 C119,C106 C121 C112,False,False,both miss (in top 10)
4,Q05,negation,What is explicitly prohibited when processing children data online?,"C205: 5 -> 5, C206: 168 -> —",C205: 0.26,C060 C035 C075,C052 C068 C035,False,False,both miss (in top 10)
5,Q06,negation,When is automated decision-making NOT allowed?,"C224: 3 -> 3, C246: 1 -> 2, C247: 21 -> —","C224: 0.74, C246: 3.93",C246 C077 C224,C077 C246 C224,True,True,both hit
6,Q07,buried_chunk,What is data protection by design and by default?,"C253: 5 -> 1, C254: 41 -> —",C253: 8.13,C085 C091 C323,C253 C085 C084,False,True,RE-RANK RESCUE
7,Q08,role_limit,Who must appoint a Data Protection Officer?,"C286: 3 -> 3, C287: 1 -> 1, C288: 2 -> 2","C286: 5.35, C287: 6.86, C288: 5.38",C287 C288 C286,C287 C288 C286,True,True,both hit
8,Q09,buried_chunk,What information must be provided when a person requests access to their data?,"C232: 4 -> 3, C233: 24 -> —, C234: 20 -> —",C232: 2.48,C059 C066 C063,C066 C064 C232,False,True,RE-RANK RESCUE
9,Q10,buried_chunk,What penalties exist for serious GDPR breaches?,"C356: 32 -> —, C415: 26 -> —, C416: 19 -> —, C417: 36 -> —",—,C092 C418 C274,C418 C420 C094,False,False,both miss (not in top 10)


In [14]:
print("Hit rate = fraction of queries whose critical chunk is among the 3 chunks given to the LLM")
print(f"  baseline : {summary['critical in baseline top3'].sum()}/{len(summary)}")
print(f"  re-rank  : {summary['critical in rerank top3'].sum()}/{len(summary)}")
print(f"  ceiling  : {(pd.Series([r['best_dense_rank'] for r in runs]) <= TOP_K).sum()}/{len(summary)}  (critical chunk inside the dense top 10 at all)")
print(summary["outcome"].value_counts().to_string())

Hit rate = fraction of queries whose critical chunk is among the 3 chunks given to the LLM
  baseline : 7/16
  re-rank  : 11/16
  ceiling  : 13/16  (critical chunk inside the dense top 10 at all)
outcome
both hit                     7
RE-RANK RESCUE               4
both miss (not in top 10)    3
both miss (in top 10)        2


### 8.2 Full Top-10 tables for the failure cases

For each query where the outcome differs between the pipelines, the complete candidate list with both ranks. `★` marks a critical chunk.

In [15]:
def top10_table(r):
    rows = []
    for c in sorted(r["ranked"], key=lambda c: c["dense_rank"]):
        ch = chunk_by_id[c["chunk_id"]]
        rows.append({"": "★" if c["chunk_id"] in r["critical"] else "", "chunk": c["chunk_id"], "section": ch["section"][:45],
                     "page": ch["page_start"], "dense rank": c["dense_rank"], "cosine": round(c["dense_score"], 3),
                     "CE score": round(c["ce_score"], 2), "rerank rank": c["rerank_rank"],
                     "shift": f"#{c['dense_rank']} → #{c['rerank_rank']}", "text": ch["text"][:90] + "…"})
    return pd.DataFrame(rows)

from IPython.display import display, Markdown
interesting = [r for r in runs if outcome(r) in ("RE-RANK RESCUE", "RE-RANK LOSS")]
for r in interesting:
    display(Markdown(f"**{r['qid']} ({r['category']}) – {outcome(r)}:** _{r['query']}_"))
    display(top10_table(r))

**Q01 (keyword_trap) – RE-RANK RESCUE:** _Can companies share user passwords with third parties?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C107,Recitals,19,1,0.260,-9.35,2,#1 → #2,(102) This Regulation is without prejudice to international agreements concluded between …
1,,C059,Recitals,11,2,0.258,-11.25,6,#2 → #6,"However, the controller should not refuse to take additional information provided by the d…"
2,,C005,Recitals,2,3,0.238,-9.66,3,#3 → #3,(6) Rapid technological developments and globalisation have brought new challenges for the…
3,,C010,Recitals,3,4,0.237,-11.27,7,#4 → #7,(13) In order to ensure a consistent level of protection for natural persons throughout th…
4,★,C187,Article 4 -- Definitions,33,5,0.235,-7.24,1,#5 → #1,"However, public authorities which may receive personal data in the 4.5.2016 EN framework o…"
5,,C115,Recitals,21,6,0.221,-11.22,5,#6 → #5,Controllers and processors should be encouraged to provide additional safeguards via contr…
6,,C334,Article 50 -- International cooperation for t,65,7,0.220,-9.87,4,#7 → #4,Article 50 -- International cooperation for the protection of personal data In relation to…
7,,C004,Recitals,2,8,0.217,-11.30,9,#8 → #9,This Regulation respects all fundamental rights and observes the freedoms and principles r…
8,,C270,Article 31 -- Cooperation with the supervisor,51,9,0.215,-11.37,10,#9 → #10,Article 31 -- Cooperation with the supervisory authority The controller and the processor …
9,,C231,Article 14 -- Information to be provided wher,42,10,0.212,-11.29,8,#10 → #8,In such cases the controller shall take appropriate measures to protect the data subject's…


**Q02 (role_limit) – RE-RANK RESCUE:** _Can a data processor hire another subcontractor without permission?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C265,Article 29 -- Processing under the authority,50,1,0.485,-5.86,6,#1 → #6,Article 29 -- Processing under the authority of the controller or processor The processor …
1,,C127,Recitals,23,2,0.439,-4.40,4,#2 → #4,"For that purpose, the supervisory authorities should cooperate with each other and with th…"
2,,C033,Recitals,7,3,0.435,-7.67,9,#3 → #9,That criterion should not depend on whether the processing of personal data is carried out…
3,★,C259,Article 28 -- Processor,49,4,0.430,0.59,1,#4 → #1,Article 28 -- Processor 1. Where processing is to be carried out on behalf of a controller…
4,,C114,Recitals,20,5,0.425,-0.41,2,#5 → #2,Authorisation by the competent supervisory authority should be obtained when the safeguard…
5,,C270,Article 31 -- Cooperation with the supervisor,51,6,0.421,-8.09,10,#6 → #10,Article 31 -- Cooperation with the supervisory authority The controller and the processor …
6,,C290,Article 38 -- Position of the data protection,56,7,0.412,-6.57,8,#7 → #8,Data subjects may contact the data protection officer with regard to all issues related to…
7,,C262,Article 28 -- Processor,49,8,0.407,-1.19,3,#8 → #3,(h) makes available to the controller all information necessary to demonstrate compliance…
8,,C288,Article 37 -- Designation of the data protect,55,9,0.407,-5.61,5,#9 → #5,"The data protection officer may be a staff member of the controller or processor, or fulfi…"
9,,C324,Article 47 -- Binding corporate rules,63,10,0.393,-6.39,7,#10 → #7,(e) the rights of data subjects in regard to processing and the means to exercise those r…


**Q07 (buried_chunk) – RE-RANK RESCUE:** _What is data protection by design and by default?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C085,Recitals,15,1,0.679,6.80,2,#1 → #2,"When developing, designing, selecting and using applications, services and products that a…"
1,,C091,Recitals,16,2,0.653,-3.67,6,#2 → #6,"Those measures should ensure an appropriate level of security, including confidentiality, …"
2,,C323,Article 47 -- Binding corporate rules,63,3,0.638,3.75,4,#3 → #4,"(a) the structure and contact details of the group of undertakings, or group of enterpris…"
3,,C084,Recitals,15,4,0.627,6.63,3,#4 → #3,(78) The protection of the rights and freedoms of natural persons with regard to the proce…
4,★,C253,Article 25 -- Data protection by design and b,48,5,0.625,8.13,1,#5 → #1,Article 25 -- Data protection by design and by default 1. Taking into account the state of…
5,,C278,Article 35 -- Data protection impact assessme,53,6,0.605,-3.79,7,#6 → #7,Article 35 -- Data protection impact assessment 1. Where a type of processing in particula…
6,,C319,Article 46 -- Transfers subject to appropriat,62,7,0.605,-4.12,8,#7 → #8,(d) standard data protection clauses adopted by a supervisory authority and approved by t…
7,,C292,Article 39 -- Tasks of the data protection of,56,8,0.604,-5.11,9,#8 → #9,(c) to provide advice where requested as regards the data protection impact assessment an…
8,,C250,Article 23 -- Restrictions,47,9,0.600,-6.16,10,#9 → #10,(i) the protection of the data subject or the rights and freedoms of others; (j) the enf…
9,,C097,Recitals,17,10,0.593,-3.33,5,#10 → #5,"(90) In such cases, a data protection impact assessment should be carried out by the contr…"


**Q09 (buried_chunk) – RE-RANK RESCUE:** _What information must be provided when a person requests access to their data?_

,,chunk,section,page,dense rank,cosine,CE score,rerank rank,shift,text
0,,C059,Recitals,11,1,0.703,0.98,8,#1 → #8,"However, the controller should not refuse to take additional information provided by the d…"
1,,C066,Recitals,12,2,0.690,3.53,1,#2 → #1,"Where possible, the controller should be able to provide remote access to a secure system …"
2,,C063,Recitals,12,3,0.682,1.56,5,#3 → #5,"Where the icons are presented electronically, they should be machine-readable. (61) The in…"
3,★,C232,Article 15 -- Right of access by the data sub,43,4,0.674,2.48,3,#4 → #3,Article 15 -- Right of access by the data subject 1. The data subject shall have the right…
4,,C064,Recitals,12,5,0.671,2.63,2,#5 → #2,Where the origin of the personal data cannot be provided to the data subject because vario…
5,,C065,Recitals,12,6,0.660,2.24,4,#6 → #4,"In that regard, the number of data subjects, the age of the data and any appropriate safeg…"
6,,C231,Article 14 -- Information to be provided wher,42,7,0.647,-4.07,10,#7 → #10,In such cases the controller shall take appropriate measures to protect the data subject's…
7,,C027,Recitals,6,8,0.642,1.10,7,#8 → #7,"This may leave traces which, in particular when combined with unique identifiers and other…"
8,,C276,Article 34 -- Communication of a personal dat,53,9,0.640,-2.77,9,#9 → #9,The communication to the data subject referred to in paragraph 1 shall not be required if …
9,,C225,Article 14 -- Information to be provided wher,41,10,0.626,1.32,6,#10 → #6,Article 14 -- Information to be provided where personal data have not been obtained from t…


### 8.3 Answer 1 (baseline) vs. Answer 2 (re-ranked)

Both answers are generated for every query. When both pipelines select the same three chunks (in any order) the prompts differ only in passage order.

In [16]:
answers = []
for r in runs:
    p1 = build_prompt(r["query"], r["base_top"])
    p2 = build_prompt(r["query"], r["rr_top"])
    a1 = generate(p1)
    a2 = generate(p2)
    chk = lambda a: None if (a["text"] is None or not r.get("answer_check")) else bool(re.search(r["answer_check"], a["text"], re.IGNORECASE))
    answers.append({"qid": r["qid"], "query": r["query"], "outcome": outcome(r),
                    "same_chunk_set": {c["chunk_id"] for c in r["base_top"]} == {c["chunk_id"] for c in r["rr_top"]},
                    "answer1": a1["text"], "answer2": a2["text"],
                    "answer1_has_key_fact": chk(a1), "answer2_has_key_fact": chk(a2),
                    "llm_latency1_s": a1["latency_s"], "llm_latency2_s": a2["latency_s"],
                    "model_version": a1.get("model_version") or a2.get("model_version"),
                    "from_cache": [a1.get("cached"), a2.get("cached")],
                    "errors": [e for e in (a1.get("error"), a2.get("error")) if e]})

with open(RESULTS_DIR / "answers.json", "w") as f:
    json.dump(answers, f, indent=2, ensure_ascii=False)
ans_df = pd.DataFrame(answers)
print("LLM model version reported by API:", ans_df["model_version"].dropna().unique(), "| failed calls:", int(ans_df["errors"].map(len).sum()))
ans_df[["qid", "outcome", "same_chunk_set", "answer1_has_key_fact", "answer2_has_key_fact"]]

LLM model version reported by API: ['gemini-3.1-flash-lite'] | failed calls: 0


,qid,outcome,same_chunk_set,answer1_has_key_fact,answer2_has_key_fact
0,Q01,RE-RANK RESCUE,False,False,False
1,Q02,RE-RANK RESCUE,False,False,True
2,Q03,both miss (not in top 10),False,False,False
3,Q04,both miss (in top 10),False,True,True
4,Q05,both miss (in top 10),False,False,False
5,Q06,both hit,True,False,True
6,Q07,RE-RANK RESCUE,False,False,True
7,Q08,both hit,True,True,True
8,Q09,RE-RANK RESCUE,False,False,True
9,Q10,both miss (not in top 10),False,True,True


In [17]:
# Print answers side by side for the queries where the selected chunks differ.
for a in answers:
    if a["same_chunk_set"]:
        continue
    display(Markdown(f"---\n**{a['qid']} – {a['outcome']}** — _{a['query']}_\n\n"
                     f"**Answer 1 (baseline):** {a['answer1']}\n\n**Answer 2 (re-ranked):** {a['answer2']}"))

---
**Q01 – RE-RANK RESCUE** — _Can companies share user passwords with third parties?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The provided context does not answer this question.

---
**Q02 – RE-RANK RESCUE** — _Can a data processor hire another subcontractor without permission?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The processor SHALL NOT engage another processor without prior specific or general written authorisation of the controller [C259]. In the case of general written authorisation, the processor SHALL inform the controller of any intended changes concerning the addition or replacement of other processors, thereby giving the controller the opportunity to object to such changes [C259].

---
**Q03 – both miss (not in top 10)** — _Is processing sensitive data ever allowed without explicit consent?_

**Answer 1 (baseline):** Yes, processing of special categories of personal data is allowed in specific cases set out in the Regulation, even beyond explicit consent [C052]. Derogations from the general prohibition for processing such data may be provided where processing is allowed in specific cases, such as when Member State law lays down specific provisions to comply with a legal obligation or for the performance of a task carried out in the public interest or in the exercise of official authority [C052]. Additionally, processing may be permitted in respect of specific needs, such as when carried out in the course of legitimate activities by certain associations or foundations [C052].

**Answer 2 (re-ranked):** Yes, processing of special categories of personal data MAY be allowed without explicit consent in specific cases set out in the Regulation, such as when Member States law lays down specific provisions to adapt rules for compliance with a legal obligation, the performance of a task carried out in the public interest, or the exercise of official authority [C052]. Additionally, derogations from the general prohibition for processing such data MAY apply in respect of specific needs, such as when processing is carried out in the course of legitimate activities by certain associations or foundations [C052].

---
**Q04 – both miss (in top 10)** — _Can a controller transfer personal data outside the EU?_

**Answer 1 (baseline):** Yes, a controller MAY transfer personal data to a third country or international organization under specific conditions, such as when the Commission has decided on an adequate level of protection or when using solutions that provide enforceable and effective rights [C120]. In cases where no other grounds for transfer are applicable, a transfer MAY take place if it is not repetitive, concerns a limited number of data subjects, is necessary for compelling legitimate interests, and the controller has provided suitable safeguards [C331]. Such transfers SHOULD be possible only in residual cases where no other grounds for transfer are applicable [C119]. The controller SHALL inform the supervisory authority and the data subject about the transfer [C119, C331].

**Answer 2 (re-ranked):** Yes, a controller MAY transfer personal data to third countries or international organisations, provided that the conditions laid down in the Regulation are fully complied with [C106]. Transfers SHOULD only be allowed where the conditions of the Regulation for a transfer to third countries are met [C121]. If an adequacy decision is absent, the controller or processor SHOULD take measures to compensate for the lack of data protection by way of appropriate safeguards for the data subject [C112].

---
**Q05 – both miss (in top 10)** — _What is explicitly prohibited when processing children data online?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The provided context does not answer this question.

...

---
**Q12 – both miss (not in top 10)** — _Can personal data be processed for scientific research without consent?_

**Answer 1 (baseline):** The provided context does not answer this question.

**Answer 2 (re-ranked):** The provided context does not answer this question.

---
**Q13 – both hit** — _What rights can be restricted under national law?_

**Answer 1 (baseline):** Union or Member State law MAY restrict the scope of the obligations and rights provided for in Articles 12 to 22 and Article 34, as well as Article 5 in so far as its provisions correspond to the rights and obligations provided for in Articles 12 to 22 [C248]. Such restrictions are permitted when they respect the essence of fundamental rights and freedoms and are a necessary and proportionate measure in a democratic society to safeguard specific interests like national security, defence, public security, or the prevention and prosecution of criminal offences [C248].

**Answer 2 (re-ranked):** Union or Member State law MAY restrict the scope of the obligations and rights provided for in Articles 12 to 22 and Article 34, as well as Article 5 in so far as its provisions correspond to the rights and obligations provided for in Articles 12 to 22 [C248]. Such restrictions are permitted when they respect the essence of fundamental rights and freedoms and are a necessary and proportionate measure in a democratic society to safeguard specific interests like national security, defence, public security, or the prevention and investigation of criminal offences [C248].

---
**Q14 – both hit** — _What breach notification obligations apply to processors?_

**Answer 1 (baseline):** The processor SHALL notify the controller without undue delay after becoming aware of a personal data breach [C273].

**Answer 2 (re-ranked):** The processor SHALL notify the controller without undue delay after becoming aware of a personal data breach [C273].

---
**Q15 – both hit** — _Under what conditions can a supervisory authority impose a temporary ban on processing?_

**Answer 1 (baseline):** A supervisory authority has the corrective power to impose a temporary or definitive limitation, including a ban on processing [C354], [C132]. Additionally, the supervisory authority may exercise its power to prohibit processing operations even if it does not respond to a request for consultation regarding a data protection impact assessment within a specified period [C101].

**Answer 2 (re-ranked):** The powers of supervisory authorities, including the power to impose a temporary or definitive limitation or ban on processing, SHOULD be exercised in accordance with appropriate procedural safeguards set out in Union and Member State law, impartially, fairly, and within a reasonable time [C133]. In particular, each measure SHOULD be appropriate, necessary, and proportionate in view of ensuring compliance with the Regulation, taking into account the circumstances of each individual case [C133]. Furthermore, the authority must respect the right of every person to be heard before any individual measure which would affect them adversely is taken, and avoid superfluous costs and excessive inconveniences for the persons concerned [C133].

---
**Q16 – both hit** — _Do users have the right to delete their account data?_

**Answer 1 (baseline):** The data subject SHALL have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay where specific grounds apply, such as when the data are no longer necessary for the purposes for which they were collected or when consent is withdrawn [C236]. Additionally, a data subject SHOULD have the right to have personal data erased where the retention of such data infringes the Regulation or applicable law [C067]. The provided context does not explicitly use the term "account data," but it establishes the right to erasure for personal data under the conditions specified [C236].

**Answer 2 (re-ranked):** The data subject SHALL have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay where specific grounds apply, such as when the data is no longer necessary for the purposes for which it was collected, consent is withdrawn, or the data subject objects to the processing [C236]. Additionally, a data subject SHOULD have the right to have personal data erased where the retention of such data infringes the Regulation or applicable law [C067]. However, the further retention of personal data SHOULD be lawful if it is necessary for reasons such as exercising the right of freedom of expression, compliance with a legal obligation, or the establishment, exercise, or defence of legal claims [C068].

## 9. Latency profiling

**Measurement boundaries** (wall-clock, `time.perf_counter`, CPU only, models already loaded, after warm-up):

* **Baseline retrieval** = encode query with bi-encoder + FAISS search for Top 10 + take first 3.
* **Re-rank pipeline retrieval** = the same dense step **plus** cross-encoder scoring of the 10 pairs + sort + take first 3.
* **LLM generation** is timed separately (one Gemini call per answer, network + server time, measured when the call was made and stored in the cache) because it is dominated by the API and is the same kind of work in both pipelines.

Each query is timed `REPEATS` times; the table reports the median and 95th percentile over all query × repeat measurements.

In [18]:
REPEATS = 20

def time_once(q):
    t0 = time.perf_counter()
    cands = dense_retrieve(q); _ = baseline_select(cands)
    t1 = time.perf_counter()
    _ = rerank(q, cands)
    t2 = time.perf_counter()
    return t1 - t0, t2 - t1

for qd in QUERIES[:3]:                              # warm-up (thread pools, caches)
    time_once(qd["query"])

dense_t, ce_t = [], []
for _ in range(REPEATS):
    for qd in QUERIES:
        d, c = time_once(qd["query"])
        dense_t.append(d); ce_t.append(c)
dense_ms, ce_ms = np.array(dense_t) * 1000, np.array(ce_t) * 1000
total_ms = dense_ms + ce_ms

llm1 = [a["llm_latency1_s"] for a in answers if a["llm_latency1_s"] is not None]
llm2 = [a["llm_latency2_s"] for a in answers if a["llm_latency2_s"] is not None]
stat = lambda x: (round(float(np.median(x)), 1), round(float(np.percentile(x, 95)), 1), round(float(np.mean(x)), 1))
rows = [
    ("Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3)", *stat(dense_ms), len(dense_ms)),
    ("Re-ranking step only (cross-encoder on 10 pairs + sort)",      *stat(ce_ms), len(ce_ms)),
    ("Pipeline 2: dense retrieval + re-ranking",                     *stat(total_ms), len(total_ms)),
]
if llm1 and llm2:
    rows += [("LLM generation, Answer 1 (Gemini API)", *stat(np.array(llm1) * 1000), len(llm1)),
             ("LLM generation, Answer 2 (Gemini API)", *stat(np.array(llm2) * 1000), len(llm2))]
latency = pd.DataFrame(rows, columns=["stage", "median ms", "p95 ms", "mean ms", "n"])
latency.to_csv(RESULTS_DIR / "latency.csv", index=False)
print(f"Re-ranking adds {np.median(ce_ms):.1f} ms median ({np.median(total_ms) / np.median(dense_ms):.1f}x the baseline retrieval time).")
latency

Re-ranking adds 424.0 ms median (21.8x the baseline retrieval time).


,stage,median ms,p95 ms,mean ms,n
0,Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3),20.4,21.6,17.1,320
1,Re-ranking step only (cross-encoder on 10 pairs + sort),424.0,437.8,391.8,320
2,Pipeline 2: dense retrieval + re-ranking,443.8,459.0,408.8,320
3,"LLM generation, Answer 1 (Gemini API)",1351.1,6807.1,2412.7,16
4,"LLM generation, Answer 2 (Gemini API)",1330.0,7391.3,2269.0,16


### Scaling: why not run the cross-encoder over the whole collection?

To ground the scaling question in measurements, the cross-encoder is timed on all chunks of this document for one query and compared with scoring 10 candidates. The per-pair cost is then extrapolated linearly to larger collections (an **extrapolation**, labelled as such). The bi-encoder side only needs one query embedding at query time because chunk embeddings were computed offline (§4).

In [19]:
q = QUERIES[0]["query"]
all_pairs = [(q, c["text"]) for c in chunks]
cross_encoder.predict(all_pairs[:32], batch_size=32, show_progress_bar=False)       # warm-up
full_times = []
for _ in range(3):
    t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32, show_progress_bar=False)
    full_times.append(time.perf_counter() - t0)
full_s = float(np.median(full_times))
per_pair_ms = full_s * 1000 / len(all_pairs)
emb_per_chunk_ms = index_build_s * 1000 / len(chunks)

scale = pd.DataFrame([{
    "collection size (chunks)": n,
    "CE on all chunks per query (s)": round(n * per_pair_ms / 1000, 2),
    "dense search + CE on top-10 per query (ms)": round(float(np.median(total_ms)), 1) if n == len(chunks) else "≈ same (+ ANN search)",
    "one-off offline embedding cost (s)": round(n * emb_per_chunk_ms / 1000, 1),
} for n in [len(chunks), 10_000, 1_000_000]])
print(f"measured: CE over all {len(chunks)} chunks = {full_s*1000:.0f} ms per query ({per_pair_ms:.2f} ms/pair); "
      f"CE over 10 candidates = {np.median(ce_ms):.1f} ms")
scale.to_csv(RESULTS_DIR / "scaling.csv", index=False)
scale

measured: CE over all 444 chunks = 18219 ms per query (41.03 ms/pair); CE over 10 candidates = 424.0 ms


,collection size (chunks),CE on all chunks per query (s),dense search + CE on top-10 per query (ms),one-off offline embedding cost (s)
0,444,18.22,443.8,10.2
1,10000,410.33,≈ same (+ ANN search),229.7
2,1000000,41032.75,≈ same (+ ANN search),22966.1


## 10. Conclusions and limitations

The cell below prints the measured numbers this section is based on; the interpretation is in the report (`report/report.md`), which references these saved results (`results/`).

**Limitations (apply to every result above):**
* Small evaluation set (16 hand-written queries on one document) – the counts are descriptive, not statistically significant.
* Relevance labels are key-phrase matches chosen by me; a chunk can be relevant without containing the phrase (and vice versa). The 'answer_check' regex is a coarse screen and the answers were also read manually.
* Queries were written after reading the document and refined while exploring retrieval, so they are targeted stress tests, not a random sample of real user questions.
* The re-ranker can only reorder what the bi-encoder retrieved: when the critical chunk is outside the dense Top 10, neither pipeline can recover it.
* Both encoders are small MiniLM models (fast on CPU, not state of the art). Gemini output can vary slightly between runs even at temperature 0; answers are cached in `results/llm_cache.json`, so re-running the notebook reproduces the committed answers (delete the cache to regenerate).
* Latency was measured on one CPU laptop; LLM latency depends on network and API load.

In [20]:
print(summary[["qid", "category", "outcome"]].to_string(index=False))
print()
print(latency.to_string(index=False))

qid     category                   outcome
Q01 keyword_trap            RE-RANK RESCUE
Q02   role_limit            RE-RANK RESCUE
Q03    exception both miss (not in top 10)
Q04    exception     both miss (in top 10)
Q05     negation     both miss (in top 10)
Q06     negation                  both hit
Q07 buried_chunk            RE-RANK RESCUE
Q08   role_limit                  both hit
Q09 buried_chunk            RE-RANK RESCUE
Q10 buried_chunk both miss (not in top 10)
Q11 keyword_trap                  both hit
Q12    exception both miss (not in top 10)
Q13     negation                  both hit
Q14   role_limit                  both hit
Q15 buried_chunk                  both hit
Q16 keyword_trap                  both hit

                                                      stage  median ms  p95 ms  mean ms   n
Pipeline 1: dense retrieval (encode + FAISS top-10 + top-3)       20.4    21.6     17.1 320
    Re-ranking step only (cross-encoder on 10 pairs + sort)      424.0   437.8    39